# Evaluate — Baseline vs Trained

Produces the before/after training metrics table that goes straight into your README.

Expects these directories to exist (populated by `01_demo` + `02_train`):
- `data/baseline/random/`
- `data/baseline/heuristic/`
- `data/trained/` (optional — from notebook 02)

Runtime: CPU, ~30 seconds.

## Setup

In [ ]:
import os
REPO = 'openenv-r2-kit'
if not os.path.exists(REPO):
    !git clone -q https://github.com/Kaviyamurugadass/openenv-r2-kit.git
%cd {REPO}
!pip install -q uv
!uv sync --quiet

In [ ]:
import json
import sys; sys.path.insert(0, '.')
from pathlib import Path

from training.trajectory import TrajectoryDataset
from training.evaluation import EvaluationSuite

## 1. Load all datasets

If you haven't run notebook 02 yet, run the cell below to generate baseline data.

In [ ]:
# Bootstrap: generate baselines if missing
from training.rollout_collection import collect

for policy in ['random', 'heuristic']:
    path = Path(f'data/baseline/{policy}')
    if not path.exists() or not any(path.glob('trajectory_*.json')):
        print(f'Collecting {policy} baseline...')
        collect(episodes=10, policy=policy, output_dir=path, seed=42, domain_randomise=True)
    else:
        print(f'Found existing {policy} dataset.')

In [ ]:
datasets = {
    'random':    TrajectoryDataset.load_dir('data/baseline/random'),
    'heuristic': TrajectoryDataset.load_dir('data/baseline/heuristic'),
}

trained_path = Path('data/trained')
if trained_path.exists() and any(trained_path.glob('trajectory_*.json')):
    datasets['trained'] = TrajectoryDataset.load_dir(trained_path)
    print('Loaded trained dataset.')
else:
    print('No trained dataset — run notebook 02_train_grpo.ipynb first.')

## 2. Compute metrics for each dataset

In [ ]:
reports = {
    name: EvaluationSuite.full_report(ds, label=name)
    for name, ds in datasets.items()
}

for name, report in reports.items():
    print(f'\n=== {name} ===')
    print(json.dumps(report, indent=2, default=str))

## 3. Render the before/after table for README

Copy the markdown output into `README.md` under `## Before / After Training Metrics`.

In [ ]:
def pct_change(new, old):
    if old == 0:
        return '—'
    delta = (new - old) / abs(old) * 100
    return f'{delta:+.0f}%'

baseline = reports['heuristic']['online']  # use heuristic as the baseline
trained = reports.get('trained', {}).get('online')

rows = [
    ('Mean reward',       'mean_reward'),
    ('Success rate',      'success_rate'),
    ('Mean episode length', 'mean_length'),
    ('Median reward',     'median_reward'),
]

print('| Metric | Baseline | Trained | Change |')
print('|---|---:|---:|---:|')
for label, key in rows:
    b = baseline.get(key, '—')
    t = trained.get(key, '—') if trained else '—'
    if isinstance(b, float) and isinstance(t, float):
        print(f'| {label} | {b:.3f} | {t:.3f} | {pct_change(t, b)} |')
    else:
        print(f'| {label} | {b} | {t} | — |')

## 4. Behavior diagnostics

Catches mode collapse (agent spamming one action), low confidence, rule-violation creep.

In [ ]:
print('| Dataset | Action diversity | Mean confidence | Violations/step |')
print('|---|---:|---:|---:|')
for name, report in reports.items():
    b = report['behavior']
    print(
        f'| {name} | {b["action_diversity"]:.2f} | '
        f'{b["mean_confidence"]:.2f} | {b["mean_violations_per_step"]:.2f} |'
    )

## 5. Expert benchmark — upper bound

Runs the hand-authored expert trajectories. Judges see how close your trained agent gets to optimal.

In [ ]:
from training.benchmark import run_all

for r in run_all():
    print(
        f'{r.scenario_name:30s} '
        f'reward={r.total_reward:7.3f}  '
        f'steps={r.steps_taken:3d}  '
        f'match={r.match_ratio:4.2f}'
    )

## 6. Save the full report

In [ ]:
Path('data/eval_report.json').write_text(
    json.dumps(reports, indent=2, default=str), encoding='utf-8'
)
print('Saved → data/eval_report.json')

## What to do with the results

1. Paste the markdown table from section 3 into `README.md`
2. Include the expert benchmark line in your pitch: *"Trained agent reaches X% of expert reward"*
3. If behavior metrics show low action diversity, tune reward weights or add exploration bonus